In [107]:
!pip install langchain langgraph langchain_openai langchain_groq --quiet

In [108]:
import os
from google.colab import userdata
#os.environ["OPENAI_API_KEY"]"" = userdata.get('OPEN_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [109]:
# Import ChatModels
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Import Langgraph
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from IPython.display import display, Image

from typing_extensions import TypedDict, Optional
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages

In [110]:
# Initialize a LLM
#llm=ChatOpenAI(model="o1-mini")

llm = ChatGroq(model="qwen-2.5-32b")

## Specific LLM to generate code
llm_code = ChatGroq(model="qwen-2.5-coder-32b")


In [111]:
class State(TypedDict):
    topic: str
    code: Optional[str]
    feedback: Optional[str]
    iterations: int
    specialization: Optional[str]
    history: Optional[str]
    rating: Optional[str]
    code_compare: Optional[str]
    original_code: Optional[str]


### Create Nodes

Create the different needed nodes

In [112]:
coder_prompt = "You are a Coder specialized in {}.\
            Generate the code for the following query: {} \
            Output just the code and nothing else."

def coder_agent(state: State):
  """
  LLM Agent to code the given function in the specific language
  """
  print("Initial Code Done...")
  topic = state['topic']
  specialization = state['specialization']
  response = llm_code.invoke(coder_prompt.format(specialization, topic))
  return {'history' : "----Coder--- \n"+ response.content,'original_code': response.content,  'code': response.content, 'iterations': state['iterations']}

In [113]:
reviewer_prompt =  "You are Code reviewer specialized in {}.\
You need to review the given code following PEP8 guidelines and potential bugs\
Also check if docstring is added or not, if not make docstring mandatory\
and point out issues as bullet list.\
Also you will not make any changes to the code. You will only provide review on the given code.\
Code:\n {}"

def reviewer_agent(state: State):
  """
  LLM Agent to review the code
  """
  code = state['code']
  specialization = state['specialization']
  feedback = llm_code.invoke(reviewer_prompt.format(specialization, code))
  print("Review done...")
  return {'history' : "----Reviewer--- \n"+ feedback.content, "feedback": feedback.content,  "iterations":state["iterations"]+1}

In [114]:
code_improver = "You are a Coder specialized in {}.\
Improve the given code given the following guidelines. Guideline:\n {} \n \
Code:\n {} \n \
Output just the improved code and nothing else."

def handle_coder(state):
    history = state.get('history', '')
    feedback = state.get('feedback', '')
    code =  state.get('code','')
    specialization = state.get('specialization','')

    print("CODER rewriting...")
    code = llm_code.invoke(code_improver.format(specialization,feedback,code))
    return {'code':code.content}

In [115]:
rating_start = "Rate the skills of the coder on a scale of 10 given the Code review cycle with a short reason.\
Code review:\n {} \n "

code_comparison = "Compare the two code snippets and rate on a scale of 10 to both. Dont output the codes.Revised Code: \n {} \n Actual Code: \n {}"


def handle_result(state):
    print("Handling results...")

    #history = state.get('history', '').strip()
    code1 = state.get('code', '')
    code2 = state.get('original_code', '')
    #rating  = llm(rating_start.format(history))

    code_compare = llm_code.invoke(code_comparison.format(code1,code2))
    return {'code_compare':code_compare.content}

In [ ]:
main_approval = ""

In [116]:
def check_feedback_agent(state: State):
  """
  LLM Agent to check if the feedback has been adressed.
  """
  review = llm_code.invoke(f"""
  Given the following python code: \n\n {state['code']}\n\n
  And the feedback: \n\n {state["feedback"]}
  \n\n Does the code adress all the feedback? Answer 'Yes' or 'No'
  """)
  evaluation = review.content == "Yes"
  print("######################",evaluation)
  return {"evaluation":evaluation}



In [117]:
def should_continue(state: State):
    if state['iterations'] >= 3:  # Set a maximum number of iterations
        return "handle_result"
    if check_feedback_agent(state)["evaluation"]:
        return "handle_result"
    return "code_improver"

In [118]:
# Workflow
workflow = StateGraph(State)

# Add Nodes
workflow.add_node("coder", coder_agent)
workflow.add_node("reviewer", reviewer_agent)
workflow.add_node("code_improver", handle_coder)
workflow.add_node("handle_result", handle_result)

# Add Edges
workflow.add_edge(START, "coder")
workflow.add_edge("coder", "reviewer")
workflow.add_edge("code_improver", "reviewer")
workflow.add_conditional_edges(
    "reviewer",
    should_continue,
    {
        "code_improver": "code_improver",
        "handle_result": "handle_result",

    }
)

workflow.add_edge("handle_result", END)
graph = workflow.compile()

In [119]:
# Show workflow
#display(Image(graph.get_graph().draw_mermaid_png()))
graph.get_graph().print_ascii()

               +-----------+                 
               | __start__ |                 
               +-----------+                 
                      *                      
                      *                      
                      *                      
                 +-------+                   
                 | coder |                   
                 +-------+                   
                      *                      
                      *                      
                      *                      
                +----------+                 
                | reviewer |                 
                +----------+                 
              ...          ...               
             .                .              
           ..                  ..            
+---------------+         +---------------+  
| code_improver |         | handle_result |  
+---------------+         +---------------+  
                                  

In [120]:
initial_state = {
  'topic': 'Generate fibonacci series',
  'feedback': "",
  'iterations': 0,
  'specialization': 'Python',
  'history' : None
}

In [121]:
from IPython.display import Markdown

final_state = graph.invoke(initial_state)

Initial Code Done...
Review done...
###################### False
CODER rewriting...
Review done...
###################### False
CODER rewriting...
Review done...
Handling results...


```python
def fibonacci_series(n):
    """
    Print the Fibonacci series up to n terms.

    Parameters:
    n (int): The number of terms in the Fibonacci series to print. Must be a non-negative integer.

    Returns:
    None: This function does not return any value. It prints the Fibonacci series directly to the console.

    Raises:
    ValueError: If n is not a non-negative integer.

    Note:
    This function prints the Fibonacci series directly to the console.
    """
    if not isinstance(n, int) or n < 0:
        raise ValueError("The number of terms must be a non-negative integer.")
    
    a, b = 0, 1
    for _ in range(n):
        print(a, end=' ')
        a, b = b, a + b

fibonacci_series(10)
```

In [ ]:
Markdown(final_state['code'])

In [124]:
final_state['iterations']

3

In [126]:
final_state['code_compare']

'Revised Code: 9  \nActual Code: 5'

In [125]:
Markdown(final_state['feedback'])

Here is the code review based on PEP8 guidelines, potential bugs, and docstring requirements:

- **PEP8 Compliance:**
  - The code adheres to PEP8 guidelines for naming conventions and structure. The function name `fibonacci_series` is descriptive and uses snake_case.
  - The docstring is well-formatted and provides a clear description of the function, parameters, return value, exceptions, and additional notes.
  - The code uses consistent indentation and spacing, which is compliant with PEP8.

- **Potential Bugs:**
  - The function does not handle the case where `n` is `0`. Currently, it will print nothing if `n` is `0`. Depending on the desired behavior, this might be intentional, but it could be worth noting in the docstring or handling explicitly.
  - There are no other apparent bugs in the code.

- **Docstring:**
  - The docstring is present and follows the Google Python Style Guide, which is a common convention for Python docstrings. It includes a description, parameters, return value, exceptions, and additional notes.

- **Issues:**
  - If the intention is to print a newline after the series, consider adding `print()` at the end of the function.
  - If `n` being `0` should result in a specific message or behavior, update the function or docstring to reflect this.
  - Consider adding examples in the docstring to demonstrate function usage.

### Summary of Issues:
- **No major issues found in terms of PEP8 compliance and potential bugs.**
- **Docstring is present and well-documented.**
- **Consider handling the case where `n` is `0` explicitly or updating the docstring to clarify behavior.**
- **Consider adding a newline after the series for better output formatting.**
- **Consider adding usage examples in the docstring.**